# vLLM Qwen 3.5-27B Playground

이 노트북은 vLLM으로 Qwen 3.5-27B 계열 모델을 불러와서 질문-응답 생성에 바로 사용할 수 있는 최소 코드베이스입니다.

실행 전제:
- CUDA가 가능한 GPU 환경
- `vllm` 설치 완료
- 모델 가중치를 내려받을 수 있는 Hugging Face 접근 권한 또는 로컬 캐시


In [1]:
import os
import json
from dataclasses import dataclass
from typing import Any, Dict, List, Optional

import torch
from transformers import AutoTokenizer
from vllm import LLM, SamplingParams


MODEL_ID = os.environ.get("QWEN_MODEL_ID", "Qwen/Qwen3.5-27B-Instruct")
MAX_MODEL_LEN = int(os.environ.get("QWEN_MAX_MODEL_LEN", "32768"))
DEFAULT_TEMPERATURE = float(os.environ.get("QWEN_TEMPERATURE", "0.7"))
DEFAULT_TOP_P = float(os.environ.get("QWEN_TOP_P", "0.95"))
DEFAULT_MAX_TOKENS = int(os.environ.get("QWEN_MAX_TOKENS", "1024"))

print(f"Using model: {MODEL_ID}")
print(f"CUDA available: {torch.cuda.is_available()}")
print(f"GPU count: {torch.cuda.device_count()}")

ModuleNotFoundError: No module named 'vllm'

In [ ]:
def build_llm(model_id: str = MODEL_ID, tensor_parallel_size: Optional[int] = None) -> LLM:
    """Build a vLLM engine for Qwen 3.5-27B."""
    if tensor_parallel_size is None:
        tensor_parallel_size = max(torch.cuda.device_count(), 1)

    return LLM(
        model=model_id,
        tensor_parallel_size=tensor_parallel_size,
        max_model_len=MAX_MODEL_LEN,
        trust_remote_code=True,
        dtype="bfloat16",
        gpu_memory_utilization=0.90,
        enforce_eager=False,
    )


llm = build_llm()
print("vLLM engine initialized")

In [ ]:
@dataclass
class QwenRuntime:
    model_id: str = MODEL_ID
    max_model_len: int = MAX_MODEL_LEN
    temperature: float = DEFAULT_TEMPERATURE
    top_p: float = DEFAULT_TOP_P
    max_tokens: int = DEFAULT_MAX_TOKENS


runtime = QwenRuntime()
tokenizer = AutoTokenizer.from_pretrained(runtime.model_id, trust_remote_code=True)


def build_sampling_params(
    temperature: float = runtime.temperature,
    top_p: float = runtime.top_p,
    max_tokens: int = runtime.max_tokens,
) -> SamplingParams:
    return SamplingParams(
        temperature=temperature,
        top_p=top_p,
        max_tokens=max_tokens,
    )


def format_messages(system_prompt: str, user_prompt: str) -> str:
    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_prompt},
    ]
    return tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
    )


def generate_text(
    user_prompt: str,
    system_prompt: str = "You are a precise and helpful assistant.",
    temperature: float = runtime.temperature,
    top_p: float = runtime.top_p,
    max_tokens: int = runtime.max_tokens,
) -> str:
    prompt = format_messages(system_prompt, user_prompt)
    params = build_sampling_params(
        temperature=temperature,
        top_p=top_p,
        max_tokens=max_tokens,
    )
    outputs = llm.generate([prompt], params)
    return outputs[0].outputs[0].text.strip()


print("Tokenizer loaded and helpers are ready.")

In [ ]:
def generate_batch(
    prompts: List[str],
    system_prompt: str = "You are a precise and helpful assistant.",
    temperature: float = runtime.temperature,
    top_p: float = runtime.top_p,
    max_tokens: int = runtime.max_tokens,
) -> List[str]:
    formatted_prompts = [format_messages(system_prompt, prompt) for prompt in prompts]
    params = build_sampling_params(
        temperature=temperature,
        top_p=top_p,
        max_tokens=max_tokens,
    )
    outputs = llm.generate(formatted_prompts, params)
    return [output.outputs[0].text.strip() for output in outputs]


def save_qa_examples(examples: List[Dict[str, Any]], output_path: str) -> None:
    with open(output_path, "w", encoding="utf-8") as f:
        for example in examples:
            f.write(json.dumps(example, ensure_ascii=False) + "\n")


sample_question = "vLLM을 사용해서 Qwen 모델을 서빙할 때 어떤 장점이 있나요?"
sample_answer = generate_text(sample_question)
print(sample_answer)